# CAD AI Inspection — Full GPU Benchmark

Run the complete model comparison from one command. Enable a Kaggle GPU (T4 or better). The orchestrator isolates each experiment and records failures instead of stopping the entire benchmark. LocateAnything is resource-intensive and can be disabled with `--skip-locateanything`.

In [ ]:
# Kaggle setup: set PROJECT_ROOT to the extracted project directory.
from pathlib import Path
import os, subprocess, sys

# If this notebook is stored inside the project folder, this resolves automatically.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "run_all_benchmarks.py").exists():
    candidates = list(Path('/kaggle/input').rglob('run_all_benchmarks.py'))
    if not candidates:
        raise FileNotFoundError('Upload/extract the CAD_AI_Inspection project as a Kaggle Dataset first.')
    PROJECT_ROOT = candidates[0].parent.parent
print('Project:', PROJECT_ROOT)
print('GPU:')
subprocess.run([sys.executable, '-c', 'import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA GPU")'])

In [ ]:
# Install dependencies. Kaggle usually already has PyTorch; this command avoids replacing it.
%cd $PROJECT_ROOT
!pip install -q -r requirements.txt
# Optional: update Transformers if required by Qwen3-VL support.
!pip install -q -U "transformers>=4.57.0" accelerate timm

## Run all experiments

This includes the original baseline, registration, DINOv3, Florence-2, Qwen3-VL and LocateAnything. The latter may fail on a T4 because of memory requirements; that failure is recorded and does not invalidate other experiments. Model weights download on first use.

In [ ]:
DATA_ROOT = PROJECT_ROOT / 'data' / 'sample' / 'AI_sample_CAD'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'kaggle_full_benchmark'
cmd = [sys.executable, 'scripts/run_all_benchmarks.py',
       '--data', str(DATA_ROOT), '--output', str(OUTPUT_ROOT),
       '--device', 'cuda', '--max-side', '1200', '--timeout-minutes', '90']
print(' '.join(map(str, cmd)))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=False)

In [ ]:
# Inspect consolidated experiment status
import pandas as pd, json
summary_path = OUTPUT_ROOT / 'benchmark_summary.csv'
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print('No summary yet. Check individual output folders and run.log files.')
if (OUTPUT_ROOT / 'SUMMARY.json').exists():
    print((OUTPUT_ROOT / 'SUMMARY.json').read_text())

## Download results

Zip the output directory and download it from the Kaggle notebook output panel. `benchmark_runs.json` contains commands, statuses, runtime and log tails. `benchmark_summary.csv` is the quick comparison index. A completed script run is not proof of model accuracy; annotate the component-level ground truth before reporting precision, recall or F1.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT)
print('Results archive:', archive)